<h1 style="font-family: 'Times New Roman'; text-align: center;"><b>Machine Learning</b></h1>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b>URL Phishing Detection Project</b></h2>

<h3 style="font-family: 'Times New Roman'; text-align: right;"><b>- Pallav Biyala (24110234)</b></h3>

<hr>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b> Logistic Regression </b></h2>

### Importing Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,balanced_accuracy_score, recall_score
from sklearn.pipeline import Pipeline # this performs regularization
from sklearn.exceptions import ConvergenceWarning

In [2]:
df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
df1 = df.copy() # Creating a copy if required to use in future

<h3 style="font-family: 'Times New Roman';"><b> Pre-Processing of the Data</b></h3>

In [3]:
#### Removing the Duplicated URLs we found in the EDA
df = df.drop_duplicates(subset=['URL'], keep='first').copy()

### So in features, we have 50 features excluding label and rest 5 string valued features
X = df.drop(columns=["label", "FILENAME", "URL", "Domain", "TLD", "Title"], errors="ignore")

### And in y we have only lavel
y = df["label"]

# Dividng dataset into (Train+Val and test)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,y,
    test_size=0.1,
    random_state=42,
    stratify=y # making sure we have similar ratio across both classes across all two sets
)

# now, we can split the remaining 90% of train + val ito 80% train and 10% val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=1/9,
    random_state=42,
    stratify=y_train_val
)

# so now we find the statistics of training set and make our model learn them
scaler = StandardScaler()
scaler.fit(X_train)

# Now we apply transformation
X_train_scaled = scaler.transform(X_train)

# Using validation or test data to fit the scaler would leak information from those datasets into preprocessing. Thats why we scale using X_train only.
# So thats why the lines: .fit() and .transform() for X_train can be also written as: X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Testing:", X_test_scaled.shape)

Training: (188296, 50)
Validation: (23537, 50)
Testing: (23537, 50)


<hr>

<h3 style="font-family: 'Times New Roman';"><b> Training Our Model</b></h3>

In [4]:
max_itr = 1000

model = LogisticRegression(max_iter = max_itr)
model.fit(X_train_scaled, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

<h3 style="font-family: 'Times New Roman';"><b> Evaluating Our Model</b></h3>

In [5]:
## So we trained our model. lets see first few values and see how its doing
y_val_pred = model.predict(X_val_scaled)

print("First 20 actual labels:")
print(y_val.iloc[:20].to_list())

print("\nFirst 20 predicted labels:")
print(y_val_pred[:20])

First 20 actual labels:
[0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0]

First 20 predicted labels:
[0 1 0 0 0 0 1 0 1 0 0 0 0 1 0 0 1 1 0 0]


### Accuracy

In [6]:
accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {accuracy:.4%}")

Validation Accuracy: 100.0000%


In [7]:
print("Validation confusion matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred))

Validation confusion matrix:
[[10052     0]
 [    0 13485]]

Validation classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10052
           1       1.00      1.00      1.00     13485

    accuracy                           1.00     23537
   macro avg       1.00      1.00      1.00     23537
weighted avg       1.00      1.00      1.00     23537



In [8]:
coefficients = pd.Series(
    model.coef_[0],
    index=X.columns
)

print(
    coefficients.abs()
    .sort_values(ascending=False)
    .head(15)
)

URLSimilarityIndex       7.045873
IsHTTPS                  3.947398
SpacialCharRatioInURL    1.855526
LetterRatioInURL         1.696826
NoOfJS                   1.661145
NoOfSelfRef              1.562532
HasSocialNet             1.501054
NoOfExternalRef          1.333775
NoOfImage                1.330662
DegitRatioInURL          1.299981
LineOfCode               1.163107
NoOfSubDomain            1.117006
DomainLength             1.072204
LargestLineLength        1.026840
HasCopyrightInfo         0.944610
dtype: float64


In [9]:
print(
    df.groupby("label")["URLSimilarityIndex"]
      .agg(["count", "mean", "std", "min", "median", "max"])
)

        count        mean        std         min      median    max
label                                                              
0      100520   49.651466  22.578566    0.155574   51.478561  100.0
1      134850  100.000000   0.000000  100.000000  100.000000  100.0


In [10]:
print(
    pd.crosstab(
        df["label"],
        df["URLSimilarityIndex"] >= 99.9,
        normalize="index"
    )
)

URLSimilarityIndex     False     True 
label                                 
0                   0.992181  0.007819
1                   0.000000  1.000000


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

# Convert scaled arrays back into DataFrames with original column names
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=X_val.columns)

leakage_audit = {}

for col in X_train.columns:
    # Fit Logistic Regression on a single feature at a time
    single_feat_model = LogisticRegression(max_iter=1000, random_state=42)
    single_feat_model.fit(X_train_scaled_df[[col]], y_train)
    
    # Predict and evaluate on validation set
    preds = single_feat_model.predict(X_val_scaled_df[[col]])
    acc = accuracy_score(y_val, preds)
    
    if acc > 0.80:  # Flag features achieving >80% standalone validation accuracy
        leakage_audit[col] = acc

# Create summary table
audit_df = pd.DataFrame(list(leakage_audit.items()), columns=['Feature', 'Standalone_Val_Acc'])
print(audit_df.sort_values(by='Standalone_Val_Acc', ascending=False).to_string(index=False))

           Feature  Standalone_Val_Acc
URLSimilarityIndex            0.996474
        LineOfCode            0.949994
   NoOfExternalRef            0.947275
       NoOfSelfRef            0.936016
         NoOfImage            0.927263
            NoOfJS            0.902791
           NoOfCSS            0.880486
      HasSocialNet            0.880359
  HasCopyrightInfo            0.865148
    HasDescription            0.828100


In [12]:
# 1. Drop the leaky target proxy
X_clean = df.drop(columns=['label', 'FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'URLSimilarityIndex'])
y = df['label']

# 2. Re-split 80:10:10
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_clean, y, test_size=0.10, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(1/9), random_state=42, stratify=y_train_val
)

# 3. Fit scaler and baseline Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# 4. Check clean baseline metrics
y_val_pred = model.predict(X_val_scaled)
print("Clean Baseline Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

Clean Baseline Accuracy: 0.9995326507201427

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     10052
           1       1.00      1.00      1.00     13485

    accuracy                           1.00     23537
   macro avg       1.00      1.00      1.00     23537
weighted avg       1.00      1.00      1.00     23537



In [13]:
# Define strictly static URL string features available pre-click / instantly
static_url_features = [
    'URLLength', 'DomainLength', 'TLDLength', 'NoOfSubDomain',
    'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL',
    'IsDomainIP', 'HasObfuscation', 'NoOfObfuscatedChar',
    'ObfuscationRatio', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb'
]

# Filter X to static features only
X_static = df[static_url_features]
y = df['label']

# Re-split 80:10:10
X_train_val_s, X_test_s, y_train_val_s, y_test_s = train_test_split(
    X_static, y, test_size=0.10, random_state=42, stratify=y
)
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    X_train_val_s, y_train_val_s, test_size=(1/9), random_state=42, stratify=y_train_val_s
)

# Scale features on Train set
scaler_static = StandardScaler()
X_train_s_scaled = scaler_static.fit_transform(X_train_s)
X_val_s_scaled = scaler_static.transform(X_val_s)

# Train Baseline Logistic Regression
model_static = LogisticRegression(max_iter=1000, random_state=42)
model_static.fit(X_train_s_scaled, y_train_s)

# Evaluate
y_val_s_pred = model_static.predict(X_val_s_scaled)
print(f"Static URL Feature Validation Accuracy: {accuracy_score(y_val_s, y_val_s_pred):.4%}\n")
print("Classification Report:\n", classification_report(y_val_s, y_val_s_pred))

Static URL Feature Validation Accuracy: 98.0456%

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.96      0.98     10052
           1       0.97      1.00      0.98     13485

    accuracy                           0.98     23537
   macro avg       0.98      0.98      0.98     23537
weighted avg       0.98      0.98      0.98     23537



In [14]:
from sklearn.metrics import confusion_matrix, balanced_accuracy_score

print("Confusion matrix:")
print(confusion_matrix(y_val_s, y_val_s_pred))

print("Balanced accuracy:",
      balanced_accuracy_score(y_val_s, y_val_s_pred))

Confusion matrix:
[[ 9625   427]
 [   33 13452]]
Balanced accuracy: 0.9775368639249429


In [15]:
# Check standalone accuracy of static features
X_train_s_df = pd.DataFrame(X_train_s_scaled, columns=static_url_features)
X_val_s_df = pd.DataFrame(X_val_s_scaled, columns=static_url_features)

static_audit = {}
for col in static_url_features:
    m = LogisticRegression(max_iter=1000)
    m.fit(X_train_s_df[[col]], y_train_s)
    acc = accuracy_score(y_val_s, m.predict(X_val_s_df[[col]]))
    static_audit[col] = acc

pd.Series(static_audit).sort_values(ascending=False).head(10)

NoOfOtherSpecialCharsInURL    0.799932
NoOfDegitsInURL               0.761524
DegitRatioInURL               0.754472
URLLength                     0.744700
SpacialCharRatioInURL         0.739856
NoOfLettersInURL              0.732209
CharContinuationRate          0.722564
URLCharProb                   0.720185
LetterRatioInURL              0.688575
DomainLength                  0.664273
dtype: float64

In [16]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score
import pandas as pd

# Baseline using all 21 features
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_s_scaled, y_train_s)

baseline_acc = accuracy_score(
    y_val_s,
    baseline_model.predict(X_val_s_scaled)
)

ablation_results = []

for i, feature in enumerate(static_url_features):
    # Remove this feature from the scaled arrays
    X_train_reduced = X_train_s_scaled[:, [j for j in range(len(static_url_features)) if j != i]]
    X_val_reduced = X_val_s_scaled[:, [j for j in range(len(static_url_features)) if j != i]]

    # Retrain with the remaining features
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_reduced, y_train_s)

    acc = accuracy_score(
        y_val_s,
        model.predict(X_val_reduced)
    )

    ablation_results.append({
        "Feature_Removed": feature,
        "Accuracy_Without_Feature": acc,
        "Accuracy_Drop_pp": (baseline_acc - acc) * 100
    })

ablation_df = pd.DataFrame(ablation_results)
print(
    ablation_df.sort_values("Accuracy_Drop_pp", ascending=False)
    .to_string(index=False)
)

           Feature_Removed  Accuracy_Without_Feature  Accuracy_Drop_pp
          NoOfLettersInURL                  0.888176          9.228024
                 URLLength                  0.899477          8.097888
           NoOfDegitsInURL                  0.908994          7.146195
NoOfOtherSpecialCharsInURL                  0.959001          2.145558
            NoOfQMarkInURL                  0.978459          0.199686
      CharContinuationRate                  0.979097          0.135956
             NoOfSubDomain                  0.979224          0.123210
         TLDLegitimateProb                  0.979224          0.123210
        NoOfAmpersandInURL                  0.979352          0.110464
                IsDomainIP                  0.979564          0.089221
            HasObfuscation                  0.979819          0.063729
     SpacialCharRatioInURL                  0.980159          0.029740
           NoOfEqualsInURL                  0.980244          0.021243
      

### Final Analysis:

In [17]:
# Remove repeated URLs before splitting.
df = df.drop_duplicates(subset=["URL"], keep="first").copy()

# All 50 numeric input features, excluding the target.
all_numeric_features = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != "label"
]

print("Total numeric features:", len(all_numeric_features))

# M1: Strict lexical URL features.
lexical_features = [
    "URLLength", "DomainLength", "TLDLength", "NoOfSubDomain",
    "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfEqualsInURL", "NoOfQMarkInURL",
    "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL", "IsDomainIP",
    "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio"
]

# M2: Add the three derived statistical features.
derived_features = [
    "TLDLegitimateProb",
    "URLCharProb",
    "CharContinuationRate"
]

# M3: Add HTTPS and the page-dependent features.
# URLSimilarityIndex is deliberately reserved for M4.
page_features = [
    c for c in all_numeric_features
    if c not in lexical_features
    and c not in derived_features
    and c not in ["IsHTTPS", "URLSimilarityIndex"]
]

feature_sets = {
    "M1 - Lexical": lexical_features,

    "M2 - Derived": (
        lexical_features + derived_features
    ),

    "M3 - URL Features": (
        lexical_features + derived_features + ["IsHTTPS"]
    ),

    "M4 - Page-aware": (
        lexical_features
        + derived_features
        + ["IsHTTPS"]
        + page_features
    ),

    "M5 - All numeric": all_numeric_features
}

for name, features in feature_sets.items():
    missing = set(features) - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"
    assert len(features) == len(set(features)), (
        f"{name}: duplicate feature names"
    )
    print(f"{name}: {len(features)} features")

assert "URLSimilarityIndex" not in feature_sets["M4 - Page-aware"]
assert "URLSimilarityIndex" in feature_sets["M5 - All numeric"]

Total numeric features: 50
M1 - Lexical: 17 features
M2 - Derived: 20 features
M3 - URL Features: 21 features
M4 - Page-aware: 49 features
M5 - All numeric: 50 features


In [18]:
X = df[all_numeric_features]
y = df["label"]

# 80% train+validation, 10% test.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

# Split the remaining 90% into 80% train and 10% validation
# of the original dataset.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=1/9,
    random_state=42,
    stratify=y_train_val
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 188296
Validation: 23537
Test: 23537


In [19]:
results = []
confusion_matrices = {}
classification_reports = {}
trained_models = {}
fitted_scalers = {}

for name, features in feature_sets.items():

    # Select the same rows, using only this model's features.
    X_tr = X_train[features]
    X_va = X_val[features]

    # Fit scaling only on training data.
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)

    # Same baseline algorithm and settings for every model.
    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )
    model.fit(X_tr_scaled, y_train)

    # Validation predictions.
    preds = model.predict(X_va_scaled)

    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    report = classification_report(
        y_val, preds,
        labels=[0, 1],
        target_names=["Phishing (0)", "Legitimate (1)"],
        output_dict=True,
        zero_division=0
    )

    results.append({
        "Model": name,
        "Features": len(features),
        "Accuracy": accuracy_score(y_val, preds),
        "Balanced Accuracy": balanced_accuracy_score(y_val, preds),
        "Phishing Recall": report["Phishing (0)"]["recall"],
        "Legitimate Recall": report["Legitimate (1)"]["recall"],
        "Missed Phishing": int(cm[0, 1]),
        "False Alarms": int(cm[1, 0])
    })

    confusion_matrices[name] = cm
    classification_reports[name] = report
    trained_models[name] = model
    fitted_scalers[name] = scaler

results_df = pd.DataFrame(results)

print(
    results_df.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4%}".format,
            "Balanced Accuracy": "{:.4%}".format,
            "Phishing Recall": "{:.4%}".format,
            "Legitimate Recall": "{:.4%}".format
        }
    )
)

            Model  Features  Accuracy Balanced Accuracy Phishing Recall Legitimate Recall  Missed Phishing  False Alarms
     M1 - Lexical        17  97.9267%          97.5916%        95.2945%          99.8888%              473            15
     M2 - Derived        20  98.0414%          97.7487%        95.7421%          99.7553%              428            33
M3 - URL Features        21  99.6091%          99.5474%        99.1246%          99.9703%               88             4
  M4 - Page-aware        49  99.9533%          99.9503%        99.9304%          99.9703%                7             4
 M5 - All numeric        50 100.0000%         100.0000%       100.0000%         100.0000%                0             0


In [20]:
for name, cm in confusion_matrices.items():
    print(f"\n{name}")
    print("Rows = actual; columns = predicted")
    print("Order: phishing (0), legitimate (1)")
    print(cm)


M1 - Lexical
Rows = actual; columns = predicted
Order: phishing (0), legitimate (1)
[[ 9579   473]
 [   15 13470]]

M2 - Derived
Rows = actual; columns = predicted
Order: phishing (0), legitimate (1)
[[ 9624   428]
 [   33 13452]]

M3 - URL Features
Rows = actual; columns = predicted
Order: phishing (0), legitimate (1)
[[ 9964    88]
 [    4 13481]]

M4 - Page-aware
Rows = actual; columns = predicted
Order: phishing (0), legitimate (1)
[[10045     7]
 [    4 13481]]

M5 - All numeric
Rows = actual; columns = predicted
Order: phishing (0), legitimate (1)
[[10052     0]
 [    0 13485]]


### Let's use this 5 classes of dataset now for our whole project

<hr>

### Regularization:

#### Lasso:

In [21]:
### Lasso Regularization
def run_l1_experiment(name, features):
    print(f"\n{'=' * 50}", flush=True)
    print(f"Running L1: {name}", flush=True)
    print(f"Number of features: {len(features)}", flush=True)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            l1_ratio=1.0,
            C=1.0,
            solver="liblinear",
            max_iter=500,
            tol=1e-3,
            random_state=42
        ))
    ])

    # Train
    model.fit(X_train[features], y_train)

    # Validate
    predictions = model.predict(X_val[features])

    # Inspect coefficients
    coefficients = model.named_steps["classifier"].coef_[0]

    feature_results = pd.DataFrame({
        "Feature": features,
        "Coefficient": coefficients,
        "Selected": coefficients != 0
    })

    cm = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    )

    results = {
        "Model": name,
        "Total Features": len(features),
        "Retained": int(np.count_nonzero(coefficients)),
        "Eliminated": int(np.count_nonzero(coefficients == 0)),
        "Accuracy": accuracy_score(y_val, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_val, predictions),
        "Phishing Recall": recall_score(
            y_val, predictions, pos_label=0
        ),
        "Missed Phishing": int(cm[0, 1]),
        "False Alarms": int(cm[1, 0])
    }

    print("\nFeature selection:", flush=True)
    print(feature_results.to_string(index=False), flush=True)

    print("\nConfusion matrix:", flush=True)
    print(cm, flush=True)

    print("\nMetrics:", flush=True)
    print(pd.Series(results).to_string(), flush=True)

    return model, feature_results, results

In [22]:
## running for M1
m1_l1_model, m1_feature_results, m1_l1_results = (
    run_l1_experiment(
        "M1 - Lexical",
        feature_sets["M1 - Lexical"]
    )
)


Running L1: M1 - Lexical
Number of features: 17

Feature selection:
                   Feature  Coefficient  Selected
                 URLLength   304.400178      True
              DomainLength     9.231394      True
                 TLDLength    -0.204177      True
             NoOfSubDomain     0.081284      True
          NoOfLettersInURL  -236.509535      True
          LetterRatioInURL    -1.220566      True
           NoOfDegitsInURL  -104.072537      True
           DegitRatioInURL    -1.350936      True
           NoOfEqualsInURL     0.000000     False
            NoOfQMarkInURL    -9.916690      True
        NoOfAmpersandInURL    -4.357286      True
NoOfOtherSpecialCharsInURL   -33.335748      True
     SpacialCharRatioInURL    -0.123356      True
                IsDomainIP     1.533024      True
            HasObfuscation     0.113864      True
        NoOfObfuscatedChar     0.000000     False
          ObfuscationRatio     0.000000     False

Confusion matrix:
[[ 9957    9

In [23]:
m2_l1_model, m2_feature_results, m2_l1_results = (
    run_l1_experiment(
        "M2 - Derived",
        feature_sets["M2 - Derived"]
    )
)


Running L1: M2 - Derived
Number of features: 20

Feature selection:
                   Feature  Coefficient  Selected
                 URLLength   307.437931      True
              DomainLength     8.929276      True
                 TLDLength    -0.252230      True
             NoOfSubDomain     0.184375      True
          NoOfLettersInURL  -237.450228      True
          LetterRatioInURL    -1.458653      True
           NoOfDegitsInURL  -105.626070      True
           DegitRatioInURL    -0.946781      True
           NoOfEqualsInURL     0.249890      True
            NoOfQMarkInURL    -9.853129      True
        NoOfAmpersandInURL    -4.404440      True
NoOfOtherSpecialCharsInURL   -34.431399      True
     SpacialCharRatioInURL    -0.443134      True
                IsDomainIP    -0.070422      True
            HasObfuscation    -0.885565      True
        NoOfObfuscatedChar     0.258599      True
          ObfuscationRatio     1.197643      True
         TLDLegitimateProb     

In [24]:

m3_l1_model, m3_feature_results, m3_l1_results = (
    run_l1_experiment(
        "M3 - URL Features",
        feature_sets["M3 - URL Features"]
    )
)



Running L1: M3 - URL Features
Number of features: 21

Feature selection:
                   Feature  Coefficient  Selected
                 URLLength     4.897246      True
              DomainLength     7.134630      True
                 TLDLength    -0.373096      True
             NoOfSubDomain     0.471988      True
          NoOfLettersInURL   -22.130798      True
          LetterRatioInURL    -3.499590      True
           NoOfDegitsInURL   -10.782102      True
           DegitRatioInURL    -2.862710      True
           NoOfEqualsInURL     1.820125      True
            NoOfQMarkInURL     0.081453      True
        NoOfAmpersandInURL     0.093873      True
NoOfOtherSpecialCharsInURL    -5.433378      True
     SpacialCharRatioInURL    -2.272093      True
                IsDomainIP     0.851864      True
            HasObfuscation     0.056933      True
        NoOfObfuscatedChar     0.000000     False
          ObfuscationRatio     0.298056      True
         TLDLegitimateProb

In [ ]:
m4_l1_model, m4_feature_results, m4_l1_results = (
    run_l1_experiment(
        "M4 - Page-aware",
        feature_sets["M4 - Page-aware"]
    )
)


Running L1: M4 - Page-aware
Number of features: 49


In [ ]:
m5_l1_model, m5_feature_results, m5_l1_results = (
    run_l1_experiment(
        "M5 - All numeric",
        feature_sets["M5 - All numeric"]
    )
)

In [ ]:

l1_results_df = pd.DataFrame([
    m1_l1_results,
    m2_l1_results,
    m3_l1_results,
    m4_l1_results,
    m5_l1_results
])

display(
    l1_results_df.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)


#### Modifying Hyperparameters

In [ ]:
def l1_c_sweep(name, features, C_values=(0.001, 0.01, 0.1, 1, 10, 100)):
    rows = []
    coefficient_rows = []

    for C in C_values:
        print(f"\n{name} | L1 | C={C}", flush=True)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                l1_ratio=1.0,
                C=C,
                solver="liblinear",
                max_iter=20000,
                tol=1e-6,
                random_state=42
            ))
        ])

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train[features], y_train)

        converged = not any(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        predictions = model.predict(X_val[features])
        cm = confusion_matrix(y_val, predictions, labels=[0, 1])
        coefficients = model.named_steps["classifier"].coef_[0]

        retained = int(np.count_nonzero(coefficients))
        row = {
            "Feature Set": name,
            "Penalty": "L1",
            "C": C,
            "Features": len(features),
            "Retained": retained,
            "Eliminated": len(features) - retained,
            "Accuracy": accuracy_score(y_val, predictions),
            "Balanced Accuracy": balanced_accuracy_score(y_val, predictions),
            "Phishing Recall": recall_score(
                y_val, predictions, pos_label=0
            ),
            "Missed Phishing": int(cm[0, 1]),
            "False Alarms": int(cm[1, 0]),
            "Converged": converged
        }
        rows.append(row)

        coefficient_rows.append(pd.DataFrame({
            "C": C,
            "Feature": features,
            "Coefficient": coefficients,
            "Selected": coefficients != 0
        }))

        print(
            f"Accuracy={row['Accuracy']:.4%} | "
            f"Phishing recall={row['Phishing Recall']:.4%} | "
            f"Retained={retained}/{len(features)} | "
            f"Converged={converged}",
            flush=True
        )

    results = pd.DataFrame(rows)
    coefficients_df = pd.concat(coefficient_rows, ignore_index=True)

    return results, coefficients_df

In [ ]:
m1_l1_c_results, m1_l1_c_coefficients = l1_c_sweep(
    "M1 - Lexical",
    feature_sets["M1 - Lexical"]
)

display(
    m1_l1_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m2_l1_c_results, m2_l1_c_coefficients = l1_c_sweep(
    "M2 - Derived",
    feature_sets["M2 - Derived"]
)

display(
    m2_l1_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m3_l1_c_results, m3_l1_c_coefficients = l1_c_sweep(
    "M3 - URL Features",
    feature_sets["M3 - URL Features"]
)

display(
    m3_l1_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m4_l1_c_results, m4_l1_c_coefficients = l1_c_sweep(
    "M4 - Page-aware",
    feature_sets["M4 - Page-aware"]
)

display(
    m4_l1_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m5_l1_c_results, m5_l1_c_coefficients = l1_c_sweep(
    "M5 - All numeric",
    feature_sets["M5 - All numeric"]
)

display(
    m5_l1_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

#### Ridge:

In [ ]:
### Ridge Regularization
def run_l2_experiment(name, features):
    print(f"\n{'=' * 50}", flush=True)
    print(f"Running L2: {name}", flush=True)
    print(f"Number of features: {len(features)}", flush=True)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            l1_ratio=0,
            C=1.0,
            solver="lbfgs",
            max_iter=500,
            random_state=42
        ))
    ])

    # Train
    model.fit(X_train[features], y_train)

    # Validate
    predictions = model.predict(X_val[features])

    # Inspect coefficients
    coefficients = model.named_steps["classifier"].coef_[0]

    feature_results = pd.DataFrame({
        "Feature": features,
        "Coefficient": coefficients,
        "Selected": coefficients != 0
    })

    cm = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    )

    results = {
        "Model": name,
        "Total Features": len(features),
        "Retained": int(np.count_nonzero(coefficients)),
        "Eliminated": int(np.count_nonzero(coefficients == 0)),
        "Accuracy": accuracy_score(y_val, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_val, predictions),
        "Phishing Recall": recall_score(
            y_val, predictions, pos_label=0
        ),
        "Missed Phishing": int(cm[0, 1]),
        "False Alarms": int(cm[1, 0])
    }

    print("\nFeature selection:", flush=True)
    print(feature_results.to_string(index=False), flush=True)

    print("\nConfusion matrix:", flush=True)
    print(cm, flush=True)

    print("\nMetrics:", flush=True)
    print(pd.Series(results).to_string(), flush=True)

    return model, feature_results, results

In [ ]:
## running for M1
m1_l2_model, m1_feature_results, m1_l2_results = (
    run_l2_experiment(
        "M1 - Lexical",
        feature_sets["M1 - Lexical"]
    )
)

In [ ]:
m2_l2_model, m2_feature_results, m2_l2_results = (
    run_l2_experiment(
        "M2 - Derived",
        feature_sets["M2 - Derived"]
    )
)

In [ ]:

m3_l2_model, m3_feature_results, m3_l2_results = (
    run_l2_experiment(
        "M3 - URL Features",
        feature_sets["M3 - URL Features"]
    )
)


In [ ]:
m4_l2_model, m4_feature_results, m4_l2_results = (
    run_l2_experiment(
        "M4 - Page-aware",
        feature_sets["M4 - Page-aware"]
    )
)

In [ ]:
m5_l2_model, m5_feature_results, m5_l2_results = (
    run_l2_experiment(
        "M5 - All numeric",
        feature_sets["M5 - All numeric"]
    )
)

In [ ]:

l2_results_df = pd.DataFrame([
    m1_l2_results,
    m2_l2_results,
    m3_l2_results,
    m4_l2_results,
    m5_l2_results
])

display(
    l2_results_df.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)


In [ ]:
def l2_c_sweep(name, features, C_values=(0.001, 0.01, 0.1, 1, 10, 100)):
    rows = []
    coefficient_rows = []

    for C in C_values:
        print(f"\n{name} | L2 | C={C}", flush=True)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                l1_ratio=0.0,
                C=C,
                solver="lbfgs",
                max_iter=20000,
                tol=1e-6,
                random_state=42
            ))
        ])

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train[features], y_train)

        converged = not any(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        predictions = model.predict(X_val[features])
        cm = confusion_matrix(y_val, predictions, labels=[0, 1])
        coefficients = model.named_steps["classifier"].coef_[0]

        retained = int(np.count_nonzero(coefficients))
        row = {
            "Feature Set": name,
            "Penalty": "L2",
            "C": C,
            "Features": len(features),
            "Retained": retained,
            "Eliminated": len(features) - retained,
            "Accuracy": accuracy_score(y_val, predictions),
            "Balanced Accuracy": balanced_accuracy_score(y_val, predictions),
            "Phishing Recall": recall_score(
                y_val, predictions, pos_label=0
            ),
            "Missed Phishing": int(cm[0, 1]),
            "False Alarms": int(cm[1, 0]),
            "Converged": converged
        }
        rows.append(row)

        coefficient_rows.append(pd.DataFrame({
            "C": C,
            "Feature": features,
            "Coefficient": coefficients,
            "Selected": coefficients != 0
        }))

        print(
            f"Accuracy={row['Accuracy']:.4%} | "
            f"Phishing recall={row['Phishing Recall']:.4%} | "
            f"Retained={retained}/{len(features)} | "
            f"Converged={converged}",
            flush=True
        )

    results = pd.DataFrame(rows)
    coefficients_df = pd.concat(coefficient_rows, ignore_index=True)

    return results, coefficients_df

In [ ]:
m1_l2_c_results, m1_l2_c_coefficients = l2_c_sweep(
    "M1 - Lexical",
    feature_sets["M1 - Lexical"]
)

display(
    m1_l2_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m2_l2_c_results, m2_l2_c_coefficients = l2_c_sweep(
    "M2 - Derived",
    feature_sets["M2 - Derived"]
)

display(
    m2_l2_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m3_l2_c_results, m3_l2_c_coefficients = l2_c_sweep(
    "M3 - URL Features",
    feature_sets["M3 - URL Features"]
)

display(
    m3_l2_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m4_l2_c_results, m4_l2_c_coefficients = l2_c_sweep(
    "M4 - Page-aware",
    feature_sets["M4 - Page-aware"]
)

display(
    m4_l2_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

In [ ]:
m5_l2_c_results, m5_l2_c_coefficients = l2_c_sweep(
    "M5 - All numeric",
    feature_sets["M5 - All numeric"]
)

display(
    m5_l2_c_results.style.format({
        "Accuracy": "{:.4%}",
        "Balanced Accuracy": "{:.4%}",
        "Phishing Recall": "{:.4%}"
    })
)

<hr>